# Module 8: Packaging & CI/CD Delivery

**Concepts:** packages · imports · `pyproject.toml` · **Click** CLI · GitHub Actions (overview)

**Audience:** Turn scripts into installable tools your team `pip install`s; CI runs lint/test on every push.

---

Install for CLI example: `pip install click`


## 1. Module vs package

- **Module** — one `.py` file (`deploy.py`)
- **Package** — folder with `__init__.py` (e.g. `platform_tools/`)

**Absolute import:** `from platform_tools.k8s import scale`

**Relative import** (inside package): `from .k8s import scale`


In [ ]:
# Simulated package layout (conceptual)
layout = '''
mytool/
  pyproject.toml
  src/mytool/__init__.py
  src/mytool/cli.py
'''
print(layout.strip())


## 2. Minimal `pyproject.toml`

Modern Python packaging uses PEP 621 metadata + build backend (hatchling/setuptools).


In [ ]:
PYPROJECT = '''
[project]
name = "mytool"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["click>=8.0"]

[project.scripts]
mytool = "mytool.cli:main"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
'''.strip()
print(PYPROJECT)


## 3. Click — CLI with subcommands

Platform engineers ship CLIs for safe wrappers around dangerous operations.


In [ ]:
try:
    import click
except ImportError:
    click = None

if click:

    @click.group()
    def cli():
        """Cluster helper CLI"""

    @cli.command()
    @click.option("--ns", required=True, help="Kubernetes namespace")
    @click.option("--dry-run", is_flag=True)
    def rollout(ns: str, dry_run: bool):
        prefix = "[DRY-RUN] " if dry_run else ""
        click.echo(f"{prefix}rollout in {ns}")

    cli.main(args=["rollout", "--ns", "iim-lhp", "--dry-run"], standalone_mode=False)
else:
    print("Install click to run CLI demo: pip install click")


## 4. Entry points

`[project.scripts]` maps shell command `mytool` → Python callable `main()`. After `pip install -e .`, command is on PATH.


In [ ]:
print("entry point maps: mytool -> mytool.cli:main")


## 5. GitHub Actions sketch (YAML)

CI on push: checkout → setup Python → install deps → `ruff`/`pytest` → build wheel.

You will store this as `.github/workflows/ci.yml` in a real repo — not executed inside the notebook.


In [ ]:
GITHUB_CI = '''
name: ci
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -e ".[dev]"
      - run: pytest
      - run: ruff check .
'''.strip()
print(GITHUB_CI)


## 6. Capstone — design a tiny tool

1. Package `clusterctl`
2. Command `clusterctl status --name ncs57`
3. CI runs pytest

Write the Click handler signature on paper, then implement in a real repo.


In [ ]:
def status(name: str) -> int:
    print(f"cluster {name}: READY")
    return 0


status("ncs57")


---
# Exercises


**Ex 1** — List two benefits of `pip install -e .` for local dev.


In [ ]:
# TODO: write as comments or print() two strings


**Ex 2** — Add a Click option `--replicas` (int) to a fake deploy command (pseudo-code ok).


In [ ]:
# TODO


**Ex 3** — Name three steps you would put in CI before publishing to Nexus/PyPI.


In [ ]:
# TODO


---
# Solutions


In [ ]:
print("1. edits visible immediately without reinstall")
print("2. imports use src layout like production install")


In [ ]:
# sketch
# @cli.command()
# @click.option("--replicas", type=int, default=1)
# def deploy(replicas: int):
#     click.echo(f"scaling to {replicas}")
print("see comment sketch above")


In [ ]:
steps = ["lint (ruff)", "unit tests (pytest)", "build wheel/sdist"]
for i, s in enumerate(steps, 1):
    print(i, s)


**Course complete:** revisit Module 1 capstone and rebuild it as a Click CLI with tests in Module 7 style.
